In [ ]:
from dataloading import *

train_data, dev_data, test_data = load_data()

In [ ]:
import numpy as np

BATCH_DOCS = {1: 16, 2: 8, 3: 4}

test_batches = []
for n, docs in test_data.items():
    for i in range(0, len(docs), BATCH_DOCS[n]):
        test_batches.append((n, [i for i in range(i, min(i + BATCH_DOCS[n], len(docs)))]))

dev_batches = []
for n, docs in dev_data.items():
    for i in range(0, len(docs), BATCH_DOCS[n]):
        dev_batches.append((n, [j for j in range(i, min(i + BATCH_DOCS[n], len(docs)))]))

def collate(data, n, idxs):
    docs = [data[n][i] for i in idxs]
    ids  = torch.from_numpy(np.stack([d["tokens"] for d in docs])).long()
    mask = torch.from_numpy(np.stack([d["attention_mask"] for d in docs])).long()
    return ids.to(device), mask.to(device)

In [ ]:
from model_arch import *

class DocClassifier(nn.Module):
    def __init__(self, base_model, hidden_dim=1024, num_labels=100):
        super().__init__()
        self.base = base_model
        self.head = nn.Linear(hidden_dim, num_labels)

    def forward(self, input_ids, attention_mask):
        B, n, L = input_ids.shape
        ids = input_ids.reshape(B * n, L)
        mask = attention_mask.reshape(B * n, L)
        h = self.base.tok_emb(ids)
        for block in self.base.blocks:
            h = block(h)
        h = self.base.ln_f(h)
        m = mask.unsqueeze(-1).to(h.dtype)
        pooled = (h * m).sum(1) / m.sum(1).clamp(min=1)
        scores = self.head(pooled).view(B, n, -1)
        return scores.max(dim=1).values

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = LMConfig(vocab_size=50260)
base_model = ThirdDecoderLM(cfg)
model = DocClassifier(base_model)
model.load_state_dict(torch.load("checkpoint.pt", map_location=device))
model.to(device)
model.eval()

In [ ]:
@torch.no_grad()
def predict_probs(data, batches):
    ids_all, P, Y = [], [], []
    for n, idxs in batches:
        docs = [data[n][i] for i in idxs]
        ids, mask = collate(data, n, idxs)
        
        logits = model(ids, mask)
        probs = torch.sigmoid(logits).cpu().numpy()
        P.append(probs)
        
        ids_all.extend([d["id"] for d in docs])
        
        if "labels" in docs[0]:
            Y.append(np.stack([d["labels"] for d in docs]))
            
    probs_array = np.concatenate(P)
    y_array = np.concatenate(Y) if Y else None
    
    return ids_all, probs_array, y_array

In [ ]:
from sklearn.metrics import f1_score

dev_ids, dev_probs, dev_y = predict_probs(dev_data, dev_batches, "dev")

grid = [0.01, 0.05, 0.1, 0.2, 0.3, 0.5]
best_global_c = 0.05
best_global_f1 = 0.0

for c in grid:
    preds = (dev_probs >= c).astype(int)
    score = f1_score(dev_y, preds, average="macro", zero_division=0)
    if score > best_global_f1:
        best_global_f1 = score
        best_global_c = c

In [ ]:
per_label_thresholds = np.full(100, best_global_c)

for j in range(100):
    best_f1_for_class = 0.0
    if dev_y[:, j].sum() >= 10:
        for c in grid:
            class_preds = (dev_probs[:, j] >= c).astype(int)
            score = f1_score(dev_y[:, j], class_preds, zero_division=0)
            if score > best_f1_for_class:
                best_f1_for_class = score
                per_label_thresholds[j] = c

In [ ]:
test_ids, test_probs, _ = predict_probs(test_data, test_batches, "test")
test_preds = (test_probs >= per_label_thresholds).astype(int)

output = []
for doc_id, pred_row in zip(test_ids, test_preds):
    predicted_labels = sorted([int(label_id) for label_id in np.where(pred_row == 1)[0]])
    output.append({"id": doc_id, "pred": predicted_labels})

In [ ]:
import json

with open("prediction.json", "w") as f:
    json.dump(output, f)